# 🎯 Smart Customer Support Router - Mixture of Experts (MoE)

## Objective
Build a Mixture of Experts architecture that routes customer queries to specialized experts based on intent classification.

## Architecture:
- **Router**: Classifies customer intent (Technical, Billing, General)
- **Technical Expert**: Handles bug reports and technical issues
- **Billing Expert**: Handles refund and payment queries  
- **General Expert**: Handles casual inquiries

## Key Concepts:
- Intent classification with temperature=0 for consistency
- Expert selection based on router output
- Expert-specific system prompts for specialized responses
- Using Groq API for fast inference

In [1]:
%pip install python-dotenv --upgrade --quiet groq

Note: you may need to restart the kernel to use updated packages.


## 1. Setup: Import Libraries & Load API Key

Follow best practice: Never hardcode API keys. Load from environment or input securely using `getpass`.

In [2]:
# Setup: Load API Key securely
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Note: We are using llama-3.1-70b-versatile because mixtral-8x7b-32768 was deprecated.
print("✅ Groq client initialized successfully!")

✅ Groq client initialized successfully!


## 2. Define Expert Configurations

Each expert has a specialized system prompt optimized for their domain.

In [3]:
# Expert Configuration Dictionary
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert with deep expertise in software development, debugging, and system architecture.
        
Your role:
- Provide rigorous, code-focused solutions
- Explain technical concepts with precision
- Debug issues step-by-step
- Reference specific error messages and their causes
- Provide code examples when relevant
- Focus on root cause analysis

Response style: Technical, precise, solution-oriented.""",
        "temperature": 0.7
    },
    "billing": {
        "system_prompt": """You are a Billing Support Expert specializing in payment systems, subscriptions, and financial policies.
        
Your role:
- Address payment and billing concerns empathetically
- Explain pricing and billing policies clearly
- Process refund requests fairly
- Investigate charges and discrepancies
- Provide financial reassurance when appropriate
- Follow company billing guidelines

Response style: Empathetic, professional, policy-driven.""",
        "temperature": 0.7
    },
    "general": {
        "system_prompt": """You are a General Customer Support Agent providing friendly, helpful assistance.
        
Your role:
- Help with general inquiries and questions
- Provide product information
- Offer basic troubleshooting
- Direct customers to appropriate resources when needed
- Maintain a friendly, conversational tone

Response style: Friendly, helpful, conversational.""",
        "temperature": 0.7
    }
}

print("✅ Expert configurations loaded!")
print(f"Available experts: {list(MODEL_CONFIG.keys())}")

✅ Expert configurations loaded!
Available experts: ['technical', 'billing', 'general']


## 3. The Router: Intent Classification

The router uses `temperature=0` for consistent, deterministic zero-shot classification to pick exactly one expert.

In [4]:
def route_prompt(user_input: str) -> str:
    """
    Routes a user query to the appropriate expert category.
    
    Args:
        user_input: The customer's query
        
    Returns:
        Category name: "technical", "billing", or "general"
        Returns only the category word, nothing else.
    """
    
    router_system_prompt = """You are a customer support router. Your job is to classify customer queries into exactly ONE category.

Categories:
- "technical": Bug reports, errors, code issues, system problems, crashes, debugging
- "billing": Payment issues, refunds, subscriptions, charges, invoices, pricing
- "general": Everything else - greetings, product info, casual questions

IMPORTANT: Respond with ONLY the category name. No explanation. Just one word: technical, billing, or general"""
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=10,
        temperature=0,  # Deterministic routing
        messages=[
            {"role": "system", "content": router_system_prompt},
            {"role": "user", "content": user_input}
        ]
    )
    
    # Extract the category - clean up response
    category = response.choices[0].message.content.strip().lower()
    
    # Ensure valid category
    valid_categories = ["technical", "billing", "general"]
    if category not in valid_categories:
        category = "general"  # Default fallback
    
    return category


# Test the router
print("🧪 Testing the Router:")
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "What are your business hours?"
]

for query in test_queries:
    category = route_prompt(query)
    print(f"Query: {query[:50]}...")
    print(f"Route: {category}\n")

🧪 Testing the Router:
Query: My python script is throwing an IndexError on line...
Route: technical

Query: I was charged twice for my subscription this month...
Route: billing

Query: What are your business hours?...
Route: general



## 4. The Orchestrator: Process Customer Requests

Combines the router classification with the appropriate expert's response generation.

In [5]:
def process_request(user_input: str) -> dict:
    """
    Main orchestrator function that handles the entire MoE workflow.
    
    Args:
        user_input: Customer's query
        
    Returns:
        Dictionary with route, expert response, and metadata
    """
    
    # Step 1: Route the query
    category = route_prompt(user_input)
    
    # Step 2: Get expert configuration
    expert_config = MODEL_CONFIG[category]
    system_prompt = expert_config["system_prompt"]
    temperature = expert_config["temperature"]
    
    # Step 3: Get expert response
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=1024,
        temperature=temperature,  # Higher temp for creative responses
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )
    
    expert_response = response.choices[0].message.content
    
    return {
        "customer_query": user_input,
        "routed_to": category,
        "expert_type": f"{category.upper()} EXPERT",
        "response": expert_response
    }


# Display result nicely
def display_result(result: dict):
    """Pretty-print the MoE result."""
    print("=" * 70)
    print(f"🎯 ROUTED TO: {result['expert_type']}")
    print("=" * 70)
    print(f"\n📝 Customer Query:\n{result['customer_query']}\n")
    print(f"💬 {result['expert_type']} Response:\n{result['response']}")
    print("\n" + "=" * 70 + "\n")

## 5. Demo: Testing the Smart Support Router

Let's test with the provided scenarios to verify our MoE architecture works!

In [6]:
# Test Case 1: Technical Issue
print("\n🔧 TEST CASE 1: Technical Issue\n")
technical_query = "My python script is throwing an IndexError on line 5."
result_1 = process_request(technical_query)
display_result(result_1)

# Test Case 2: Billing Issue
print("\n💳 TEST CASE 2: Billing Issue\n")
billing_query = "I was charged twice for my subscription this month."
result_2 = process_request(billing_query)
display_result(result_2)

# Test Case 3: General Inquiry
print("\n❓ TEST CASE 3: General Inquiry\n")
general_query = "What are your business hours?"
result_3 = process_request(general_query)
display_result(result_3)


🔧 TEST CASE 1: Technical Issue

🎯 ROUTED TO: TECHNICAL EXPERT

📝 Customer Query:
My python script is throwing an IndexError on line 5.

💬 TECHNICAL EXPERT Response:
To resolve the issue, I'll need more information about the error message, including the line of code where the error is occurring and the code itself. However, I can provide a general outline of how to approach this issue.

An `IndexError` in Python typically occurs when you're trying to access an element in a sequence (such as a list, tuple, or string) with an index that is out of range. Here's an example of how this might look in code:

```python
# Example of an IndexError
numbers = [1, 2, 3]
print(numbers[3])  # This will throw an IndexError
```

In this example, the error occurs because the list `numbers` only has three elements (at indices 0, 1, and 2), but we're trying to access the element at index 3, which doesn't exist.

To troubleshoot the issue in your script, I recommend the following steps:

1. **Inspect the c

## Summary

This notebook demonstrates a **Mixture of Experts** design pattern. By isolating the intent routing (Temperature=0) from the creative responses (Temperature=0.7), the bot is able to accurately categorize requests while still maintaining a personality appropriate to the situation.